<a href="https://colab.research.google.com/github/gabrielcabanilla1/EXSU-500-Clinical-NLP-large-language-models-rct/blob/main/notebooks/05_final_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
DATA_VERSION = 'PubMed_20k_RCT'
PROJECT_DIR = '/content/drive/MyDrive/exsu_project'
DATA_DIR = f'{PROJECT_DIR}/prepared_data'
MODELS_DIR = f'{PROJECT_DIR}/models'
PREDICTIONS_DIR = f'{PROJECT_DIR}/predictions'
RESULTS_DIR = f'{PROJECT_DIR}/results'
CLASSES = ['BACKGROUND', 'OBJECTIVE', 'METHODS', 'RESULTS', 'CONCLUSIONS']
SEED = 42
N_BOOTSTRAP = 2000
EXPECTED_SIZES = {'train': 180040, 'dev': 30212}
EXPECTED_TEST_SIZE = 30135
ALLOW_RERUN = False
RUNS = {'position_baseline': 'Position baseline',
        'logreg_text': 'Logistic regression (sentence)',
        'logreg_context': 'Logistic regression (sentence + context)',
        'pubmedbert_text': 'PubMedBERT (sentence)',
        'pubmedbert_context': 'PubMedBERT (sentence + context)'}

In [2]:
import os
import json
import numpy as np
import pandas as pd
from google.colab import drive

drive.mount('/content/drive')
already_done = [r for r in RUNS if os.path.exists(f'{RESULTS_DIR}/test_{r}_summary.csv')]
assert not already_done or ALLOW_RERUN, (f'Test results already exist for {already_done}. The test set is evaluated '
                                         'once; set ALLOW_RERUN = True only to reproduce the same frozen models.')

def load_split(split):
    return pd.read_csv(f'{DATA_DIR}/{DATA_VERSION}_{split}.csv', dtype={'abstract_id': str}, keep_default_na=False)

train, dev, test = load_split('train'), load_split('dev'), load_split('test')
assert len(test) == EXPECTED_TEST_SIZE and set(test.label) <= set(CLASSES)
assert not set(test.abstract_id) & (set(train.abstract_id) | set(dev.abstract_id)), 'Leakage: test abstract seen before'
print(f'Test: {test.abstract_id.nunique():,} abstracts / {len(test):,} sentences')
del train, dev

Mounted at /content/drive
Test: 2,500 abstracts / 30,135 sentences


In [3]:
from sklearn.metrics import (accuracy_score, f1_score, precision_score, roc_auc_score,
                             average_precision_score, confusion_matrix)

def bootstrap_macro_f1(y_true, y_pred, abstract_ids, n_resamples=N_BOOTSTRAP, seed=SEED):
    """95% CI of macro-F1, resampling whole abstracts (the sentences of an abstract stay together)."""
    index, unique_ids = pd.factorize(pd.Series(abstract_ids))
    k = len(CLASSES)
    counts = np.zeros((len(unique_ids), k, k), dtype=np.int64)
    np.add.at(counts, (index, y_true, y_pred), 1)
    rng = np.random.default_rng(seed)
    scores = np.empty(n_resamples)
    for b in range(n_resamples):
        cm = counts[rng.integers(0, len(unique_ids), len(unique_ids))].sum(axis=0)
        denominator = cm.sum(axis=0) + cm.sum(axis=1)
        scores[b] = np.divide(2 * np.diag(cm), denominator, out=np.zeros(k), where=denominator > 0).mean()
    return np.percentile(scores, [2.5, 97.5])

def evaluate_and_save_test(run_name, model_description, input_description, df, probs,
                      fit_seconds, hardware, settings):
    probs = np.asarray(probs, dtype=float)
    assert probs.shape == (len(df), len(CLASSES)), 'probs must be (n_sentences, 5) in CLASSES order'
    k = len(CLASSES)
    y_true = pd.Categorical(df.label, categories=CLASSES).codes
    y_pred = probs.argmax(axis=1)
    one_hot = np.eye(k)[y_true]

    cm = confusion_matrix(y_true, y_pred, labels=range(k))
    tp = np.diag(cm); fn = cm.sum(axis=1) - tp; fp = cm.sum(axis=0) - tp; tn = cm.sum() - tp - fn - fp
    per_class = pd.DataFrame({
        'precision': precision_score(y_true, y_pred, labels=range(k), average=None, zero_division=0),
        'sensitivity': tp / (tp + fn),
        'specificity': tn / (tn + fp),
        'f1': f1_score(y_true, y_pred, labels=range(k), average=None, zero_division=0),
        'auroc': [roc_auc_score(one_hot[:, i], probs[:, i]) for i in range(k)],
        'auprc': [average_precision_score(one_hot[:, i], probs[:, i]) for i in range(k)],
        'prevalence': one_hot.mean(axis=0),
    }, index=pd.Index(CLASSES, name='class'))
    per_class.loc['macro average'] = per_class.mean()

    lower, upper = bootstrap_macro_f1(y_true, y_pred, df.abstract_id)
    summary = {
        'run': run_name, 'model': model_description, 'input': input_description,
        'split': 'test', 'abstracts': df.abstract_id.nunique(), 'sentences': len(df),
        'decision_rule': 'argmax',
        'accuracy': accuracy_score(y_true, y_pred),
        'macro_f1': f1_score(y_true, y_pred, labels=range(k), average='macro', zero_division=0),
        'macro_f1_ci95_lower': lower, 'macro_f1_ci95_upper': upper,
        'weighted_f1': f1_score(y_true, y_pred, labels=range(k), average='weighted', zero_division=0),
        'macro_auroc': per_class.loc['macro average', 'auroc'],
        'macro_auprc': per_class.loc['macro average', 'auprc'],
        'macro_sensitivity': per_class.loc['macro average', 'sensitivity'],
        'macro_specificity': per_class.loc['macro average', 'specificity'],
        'fit_seconds': round(float(fit_seconds), 1), 'hardware': hardware,
        'seed': SEED, 'n_bootstrap': N_BOOTSTRAP, 'settings': json.dumps(settings),
    }

    predictions = df[['abstract_id', 'sentence_idx', 'label']].copy()
    predictions['prediction'] = np.array(CLASSES)[y_pred]
    for i, c in enumerate(CLASSES):
        predictions[f'prob_{c}'] = probs[:, i]
    predictions.to_csv(f'{PREDICTIONS_DIR}/test_{run_name}.csv', index=False)
    pd.DataFrame([summary]).to_csv(f'{RESULTS_DIR}/test_{run_name}_summary.csv', index=False)
    per_class.to_csv(f'{RESULTS_DIR}/test_{run_name}_per_class.csv')
    confusion = pd.DataFrame(cm, index=pd.Index(CLASSES, name='actual'),
                             columns=pd.Index(CLASSES, name='predicted'))
    confusion.to_csv(f'{RESULTS_DIR}/test_{run_name}_confusion_matrix.csv')

    print(f"{model_description} | {input_description}")
    print(f"Test macro-F1 {summary['macro_f1']:.4f} (95% CI {lower:.4f}-{upper:.4f}) | "
          f"accuracy {summary['accuracy']:.4f} | weighted F1 {summary['weighted_f1']:.4f} | "
          f"macro AUROC {summary['macro_auroc']:.4f} | macro AUPRC {summary['macro_auprc']:.4f}")
    print(f'Saved predictions and 3 result tables for run "{run_name}"')
    print('\nPer-class metrics (one class vs. the other four; decision rule: argmax):')
    display(per_class.round(4))
    print('\nConfusion matrix (rows = actual class, columns = predicted class):')
    display(confusion)
    never_predicted = [c for i, c in enumerate(CLASSES) if (y_pred == i).sum() == 0]
    print('Classes never predicted:', never_predicted or 'none')
    return summary, per_class

In [4]:
probabilities = {}

table_path = f'{MODELS_DIR}/position_baseline_table.csv'
if os.path.exists(table_path):
    table = pd.read_csv(table_path, index_col=0)[CLASSES]
    n_bins = len(table)
    bins = np.minimum((test.position * n_bins).astype(int), n_bins - 1)
    probabilities['position_baseline'] = table.loc[bins].to_numpy()
    print(f'position_baseline: frozen table with {n_bins} bins')

import joblib
test['idx_capped'] = test.sentence_idx.clip(upper=15)
test['from_end'] = (test.n_sentences - 1 - test.sentence_idx).clip(upper=15)
for run in ('logreg_text', 'logreg_context'):
    path = f'{MODELS_DIR}/{run}.joblib'
    if os.path.exists(path):
        model = joblib.load(path)
        probs = model.predict_proba(test)
        probabilities[run] = probs[:, [list(model.classes_).index(c) for c in CLASSES]]
        print(f"{run}: frozen model, C={model.named_steps['logisticregression'].C}")

position_baseline: frozen table with 20 bins
logreg_text: frozen model, C=4
logreg_context: frozen model, C=4


In [5]:
bert_runs = [r for r in ('pubmedbert_text', 'pubmedbert_context') if os.path.exists(f'{MODELS_DIR}/{r}/run_info.json')]
if bert_runs:
    import torch
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
    from torch.utils.data import TensorDataset, DataLoader
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print('Device:', device)
    for run in bert_runs:
        model_dir = f'{MODELS_DIR}/{run}'
        info = json.load(open(f'{model_dir}/run_info.json'))
        tokenizer = AutoTokenizer.from_pretrained(model_dir)
        model = AutoModelForSequenceClassification.from_pretrained(model_dir).to(device).eval()
        if info['use_context']:
            tokens = tokenizer(list(test.text), list(test.context), truncation='longest_first',
                               max_length=info['max_length'], padding='max_length', return_tensors='pt')
        else:
            tokens = tokenizer(list(test.text), truncation=True, max_length=info['max_length'],
                               padding='max_length', return_tensors='pt')
        chunks = []
        with torch.no_grad():
            for ids, mask in DataLoader(TensorDataset(tokens['input_ids'], tokens['attention_mask']), batch_size=256):
                with torch.autocast(device, enabled=(device == 'cuda')):
                    logits = model(input_ids=ids.to(device), attention_mask=mask.to(device)).logits
                chunks.append(torch.softmax(logits.float(), dim=-1).cpu().numpy())
        probabilities[run] = np.concatenate(chunks)
        print(f'{run}: frozen model from {model_dir}')
print('Models evaluated on test:', list(probabilities))

Device: cuda


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

pubmedbert_text: frozen model from /content/drive/MyDrive/exsu_project/models/pubmedbert_text


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

pubmedbert_context: frozen model from /content/drive/MyDrive/exsu_project/models/pubmedbert_context
Models evaluated on test: ['position_baseline', 'logreg_text', 'logreg_context', 'pubmedbert_text', 'pubmedbert_context']


In [6]:
summaries = {}
for run, probs in probabilities.items():
    summaries[run], _ = evaluate_and_save_test(run, RUNS[run], 'frozen after validation', test, probs, 0, 'frozen model', {})
    print('=' * 100)

Position baseline | frozen after validation
Test macro-F1 0.6034 (95% CI 0.5977-0.6095) | accuracy 0.6856 | weighted F1 0.6674 | macro AUROC 0.9151 | macro AUPRC 0.6228
Saved predictions and 3 result tables for run "position_baseline"

Per-class metrics (one class vs. the other four; decision rule: argmax):


,precision,sensitivity,specificity,f1,auroc,auprc,prevalence
class,,,,,,,
BACKGROUND,0.4773,0.1743,0.9739,0.2553,0.8893,0.4329,0.1202
OBJECTIVE,0.4974,0.5358,0.9546,0.5159,0.9283,0.4255,0.0774
METHODS,0.7007,0.7506,0.8432,0.7248,0.8900,0.7475,0.3284
RESULTS,0.7299,0.7726,0.8640,0.7506,0.9055,0.7684,0.3223
CONCLUSIONS,0.7100,0.8414,0.9385,0.7701,0.9625,0.7396,0.1517
macro average,0.6231,0.6149,0.9149,0.6034,0.9151,0.6228,0.2000



Confusion matrix (rows = actual class, columns = predicted class):


predicted,BACKGROUND,OBJECTIVE,METHODS,RESULTS,CONCLUSIONS
actual,,,,,
BACKGROUND,631,1250,1185,16,539
OBJECTIVE,339,1250,734,9,1
METHODS,342,12,7429,2030,84
RESULTS,8,0,1254,7504,947
CONCLUSIONS,2,1,0,722,3846


Classes never predicted: none
Logistic regression (sentence) | frozen after validation
Test macro-F1 0.7612 (95% CI 0.7543-0.7677) | accuracy 0.8248 | weighted F1 0.8223 | macro AUROC 0.9601 | macro AUPRC 0.8215
Saved predictions and 3 result tables for run "logreg_text"

Per-class metrics (one class vs. the other four; decision rule: argmax):


,precision,sensitivity,specificity,f1,auroc,auprc,prevalence
class,,,,,,,
BACKGROUND,0.6552,0.6493,0.9533,0.6522,0.9487,0.6985,0.1202
OBJECTIVE,0.6762,0.5495,0.9779,0.6063,0.9484,0.6761,0.0774
METHODS,0.8780,0.9262,0.9370,0.9015,0.9758,0.9461,0.3284
RESULTS,0.8878,0.8859,0.9468,0.8869,0.9695,0.9483,0.3223
CONCLUSIONS,0.7634,0.7545,0.9582,0.7589,0.9584,0.8385,0.1517
macro average,0.7721,0.7531,0.9547,0.7612,0.9601,0.8215,0.2000


... (11 outputs left out)

,precision,sensitivity,specificity,f1,auroc,auprc,prevalence
class,,,,,,,
BACKGROUND,0.7831,0.8694,0.9671,0.8240,0.9855,0.8863,0.1202
OBJECTIVE,0.7738,0.6438,0.9842,0.7029,0.9792,0.8192,0.0774
METHODS,0.9596,0.9763,0.9799,0.9678,0.9943,0.9865,0.3284
RESULTS,0.9729,0.9567,0.9873,0.9647,0.9944,0.9893,0.3223
CONCLUSIONS,0.9730,0.9694,0.9952,0.9712,0.9985,0.9952,0.1517
macro average,0.8925,0.8831,0.9827,0.8861,0.9904,0.9353,0.2000



Confusion matrix (rows = actual class, columns = predicted class):


predicted,BACKGROUND,OBJECTIVE,METHODS,RESULTS,CONCLUSIONS
actual,,,,,
BACKGROUND,3148,410,46,4,13
OBJECTIVE,788,1502,37,6,0
METHODS,51,29,9662,143,12
RESULTS,4,0,319,9292,98
CONCLUSIONS,29,0,5,106,4431


Classes never predicted: none


In [7]:
def paired_bootstrap(run_a, run_b, y_true, preds, abstract_ids, n_resamples=N_BOOTSTRAP, seed=SEED):
    index, unique_ids = pd.factorize(pd.Series(abstract_ids))
    k = len(CLASSES)
    counts = {}
    for run in (run_a, run_b):
        counts[run] = np.zeros((len(unique_ids), k, k), dtype=np.int64)
        np.add.at(counts[run], (index, y_true, preds[run]), 1)

    def macro_f1(cm):
        denominator = cm.sum(axis=0) + cm.sum(axis=1)
        return np.divide(2 * np.diag(cm), denominator, out=np.zeros(k), where=denominator > 0).mean()

    rng = np.random.default_rng(seed)
    differences = np.empty(n_resamples)
    for b in range(n_resamples):
        sample = rng.integers(0, len(unique_ids), len(unique_ids))
        differences[b] = macro_f1(counts[run_a][sample].sum(axis=0)) - macro_f1(counts[run_b][sample].sum(axis=0))
    point = macro_f1(counts[run_a].sum(axis=0)) - macro_f1(counts[run_b].sum(axis=0))
    lower, upper = np.percentile(differences, [2.5, 97.5])
    return {'comparison': f'{run_a} - {run_b}', 'difference': point, 'ci95_lower': lower, 'ci95_upper': upper,
            'ci_excludes_zero': bool(lower > 0 or upper < 0)}

PAIRS = [('logreg_text', 'position_baseline'), ('logreg_context', 'logreg_text'),
         ('pubmedbert_text', 'logreg_text'), ('pubmedbert_context', 'pubmedbert_text'),
         ('pubmedbert_context', 'logreg_context')]

order = test.sort_values(['abstract_id', 'sentence_idx']).index
y_true = pd.Categorical(test.label, categories=CLASSES).codes[order]
preds = {r: p.argmax(axis=1)[order] for r, p in probabilities.items()}
abstract_ids = test.abstract_id.to_numpy()[order]

columns = ['macro_f1', 'macro_f1_ci95_lower', 'macro_f1_ci95_upper', 'accuracy', 'weighted_f1',
           'macro_auroc', 'macro_auprc', 'macro_sensitivity', 'macro_specificity']
final = pd.DataFrame(summaries).T[columns].astype(float)
final.insert(0, 'model', [RUNS[r] for r in final.index])
final.to_csv(f'{RESULTS_DIR}/test_model_comparison.csv')
print('FINAL TEST RESULTS (models and settings frozen on the validation split):')
display(final.round(4))

differences = pd.DataFrame([paired_bootstrap(a, b, y_true, preds, abstract_ids)
                            for a, b in PAIRS if a in preds and b in preds])
differences.to_csv(f'{RESULTS_DIR}/test_paired_differences.csv', index=False)
print('Paired bootstrap over whole test abstracts (difference in macro-F1, first minus second):')
display(differences.round(4))

FINAL TEST RESULTS (models and settings frozen on the validation split):


,model,macro_f1,macro_f1_ci95_lower,macro_f1_ci95_upper,accuracy,weighted_f1,macro_auroc,macro_auprc,macro_sensitivity,macro_specificity
position_baseline,Position baseline,0.6034,0.5977,0.6095,0.6856,0.6674,0.9151,0.6228,0.6149,0.9149
logreg_text,Logistic regression (sentence),0.7612,0.7543,0.7677,0.8248,0.8223,0.9601,0.8215,0.7531,0.9547
logreg_context,Logistic regression (sentence + context),0.8467,0.8403,0.8530,0.8955,0.8942,0.9841,0.9043,0.8412,0.9732
pubmedbert_text,PubMedBERT (sentence),0.8272,0.8207,0.8335,0.8808,0.8799,0.9801,0.8926,0.8231,0.9699
pubmedbert_context,PubMedBERT (sentence + context),0.8861,0.8799,0.8926,0.9303,0.9295,0.9904,0.9353,0.8831,0.9827


Paired bootstrap over whole test abstracts (difference in macro-F1, first minus second):


,comparison,difference,ci95_lower,ci95_upper,ci_excludes_zero
0,logreg_text - position_baseline,0.1578,0.1494,0.1657,True
1,logreg_context - logreg_text,0.0856,0.0803,0.0912,True
2,pubmedbert_text - logreg_text,0.0661,0.0607,0.0716,True
3,pubmedbert_context - pubmedbert_text,0.0589,0.0536,0.0639,True
4,pubmedbert_context - logreg_context,0.0394,0.0337,0.0452,True
